# Module 7: 完整推理流程

## 学习目标
- 串联所有组件理解完整流程
- 跟踪一个请求的生命周期
- 理解 Engine 和 Scheduler 的协作
- 掌握多进程通信机制

---

## 7.1 系统架构回顾

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          Mini-SGLang 系统架构                              │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌─────────────┐      ┌─────────────┐      ┌─────────────┐                 │
│  │  API Server │◄────►│  Tokenizer  │◄────►│  Scheduler  │                 │
│  │  (FastAPI)  │ ZMQ  │   Worker    │ ZMQ  │   + Engine  │                 │
│  └─────────────┘      └─────────────┘      └──────┬──────┘                 │
│         │                                         │                         │
│         │ HTTP                                    │ GPU                     │
│         ▼                                         ▼                         │
│  ┌─────────────┐                          ┌─────────────┐                  │
│  │    用户     │                          │  LLM Model  │                  │
│  └─────────────┘                          └─────────────┘                  │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

## 7.2 请求生命周期

让我们跟踪一个请求从用户发送到返回响应的完整过程。

In [ ]:
import torch
from dataclasses import dataclass
from typing import List, Optional
from enum import Enum

class RequestState(Enum):
    RECEIVED = "received"       # 收到请求
    TOKENIZING = "tokenizing"   # 正在分词
    QUEUED = "queued"           # 在等待队列
    PREFILLING = "prefilling"   # 正在 prefill
    DECODING = "decoding"       # 正在 decode
    FINISHED = "finished"       # 完成

@dataclass
class RequestTracer:
    """跟踪请求生命周期"""
    uid: int
    prompt: str
    state: RequestState = RequestState.RECEIVED
    input_ids: Optional[torch.Tensor] = None
    output_ids: List[int] = None
    
    def __post_init__(self):
        self.output_ids = []
        self.history = []
    
    def transition(self, new_state: RequestState, detail: str = ""):
        self.history.append((self.state, new_state, detail))
        self.state = new_state
        print(f"  [{self.uid}] {self.state.value}: {detail}")

# 创建示例请求
tracer = RequestTracer(uid=1, prompt="What is AI?")
print(f"请求创建: uid={tracer.uid}, prompt='{tracer.prompt}'")

## 7.3 Step 1: API Server 接收请求

In [ ]:
def api_server_receive(tracer: RequestTracer):
    """模拟 API Server 接收请求"""
    print("\n=== Step 1: API Server ===")
    tracer.transition(RequestState.RECEIVED, f"收到 HTTP 请求")
    
    # 创建唯一请求 ID
    print(f"  分配请求 ID: {tracer.uid}")
    
    # 发送到 Tokenizer
    print(f"  发送 TokenizeMsg 到 Tokenizer Worker (via ZMQ)")
    
    return tracer

tracer = api_server_receive(tracer)

## 7.4 Step 2: Tokenizer Worker 分词

In [ ]:
def tokenizer_process(tracer: RequestTracer):
    """模拟 Tokenizer Worker 处理"""
    print("\n=== Step 2: Tokenizer Worker ===")
    tracer.transition(RequestState.TOKENIZING, "开始分词")
    
    # 模拟分词 (实际使用 HuggingFace tokenizer)
    # "What is AI?" -> [1001, 374, 15592, 30]
    tracer.input_ids = torch.tensor([1001, 374, 15592, 30], dtype=torch.int32)
    print(f"  输入: '{tracer.prompt}'")
    print(f"  Token IDs: {tracer.input_ids.tolist()}")
    print(f"  Token 数量: {len(tracer.input_ids)}")
    
    # 创建 UserMsg
    print(f"  创建 UserMsg 并发送到 Scheduler (via ZMQ)")
    
    return tracer

tracer = tokenizer_process(tracer)

## 7.5 Step 3: Scheduler 接收并调度

In [ ]:
def scheduler_receive(tracer: RequestTracer):
    """模拟 Scheduler 接收请求"""
    print("\n=== Step 3: Scheduler 接收 ===")
    tracer.transition(RequestState.QUEUED, "加入等待队列")
    
    print(f"  接收 UserMsg")
    print(f"  input_ids: {tracer.input_ids.tolist()}")
    
    # 检查 Radix Cache 匹配
    print(f"  查询 Radix Cache...")
    cached_len = 0  # 假设没有缓存命中
    print(f"  缓存命中: {cached_len} tokens")
    print(f"  需要计算: {len(tracer.input_ids) - cached_len} tokens")
    
    # 加入 PrefillManager
    print(f"  添加到 PrefillManager 等待队列")
    
    return tracer

tracer = scheduler_receive(tracer)

## 7.6 Step 4: Prefill 阶段

In [ ]:
def prefill_phase(tracer: RequestTracer):
    """模拟 Prefill 阶段"""
    print("\n=== Step 4: Prefill 阶段 ===")
    tracer.transition(RequestState.PREFILLING, "开始 prefill")
    
    # 资源分配
    print(f"  分配 table_idx: 0")
    print(f"  分配 KV Cache pages: [100, 101, 102, 103]")
    
    # 准备 batch
    print(f"  创建 Batch (phase=prefill, size=1)")
    print(f"  准备 attention metadata")
    
    # 模型前向传播
    print(f"  \n  --- Engine.forward_batch() ---")
    print(f"  ├─ 加载 input_ids 到 GPU")
    print(f"  ├─ 模型前向传播:")
    print(f"  │   ├─ Embedding lookup")
    print(f"  │   ├─ Layer 0: RMSNorm -> Attention -> MLP")
    print(f"  │   ├─ Layer 1: RMSNorm -> Attention -> MLP")
    print(f"  │   └─ ... (共 N 层)")
    print(f"  ├─ 最终 RMSNorm")
    print(f"  └─ LM Head -> logits")
    
    # 采样
    print(f"  \n  --- Sampler.sample() ---")
    first_token = 15592  # 模拟生成的第一个 token
    tracer.output_ids.append(first_token)
    print(f"  生成第一个 token: {first_token}")
    
    # 存储 KV Cache
    print(f"  \n  KV Cache 已存储到 pages [100, 101, 102, 103]")
    
    return tracer

tracer = prefill_phase(tracer)

## 7.7 Step 5: Decode 阶段

In [ ]:
def decode_phase(tracer: RequestTracer, max_tokens: int = 10):
    """模拟 Decode 阶段"""
    print("\n=== Step 5: Decode 阶段 ===")
    tracer.transition(RequestState.DECODING, "开始 decode")
    
    # 模拟生成的 tokens
    generated_tokens = [374, 264, 10579, 315, 21075, 30]  # "is a form of intelligence."
    eos_token = 2  # EOS token ID
    
    for step, token in enumerate(generated_tokens[:max_tokens], 1):
        print(f"\n  --- Decode Step {step} ---")
        print(f"  输入: 上一个 token {tracer.output_ids[-1]}")
        
        # 分配新的 KV Cache page
        new_page = 103 + step
        print(f"  分配 page: {new_page}")
        
        # 前向传播 (使用 CUDA Graph 加速)
        print(f"  使用 CUDA Graph 执行前向传播")
        
        # 采样
        tracer.output_ids.append(token)
        print(f"  生成 token: {token}")
        
        # 检查是否完成
        if token == eos_token or step >= max_tokens:
            print(f"  检测到结束条件")
            break
        
        # 发送增量结果
        print(f"  发送 DetokenizeMsg")
    
    print(f"\n  生成完成! 共 {len(tracer.output_ids)} tokens")
    return tracer

tracer = decode_phase(tracer, max_tokens=6)

## 7.8 Step 6: 结果返回

In [ ]:
def finish_request(tracer: RequestTracer):
    """完成请求处理"""
    print("\n=== Step 6: 完成请求 ===")
    tracer.transition(RequestState.FINISHED, "请求完成")
    
    # Scheduler 释放资源
    print(f"  释放 table_idx: 0")
    print(f"  缓存完整序列到 Radix Cache")
    
    # Detokenizer 处理
    print(f"  \n  --- Detokenizer ---")
    print(f"  输入 token IDs: {tracer.output_ids}")
    
    # 模拟解码
    output_text = "AI is a form of intelligence."
    print(f"  输出文本: '{output_text}'")
    
    # API Server 返回
    print(f"  \n  --- API Server ---")
    print(f"  返回 HTTP 响应")
    print(f"  响应内容: {{'text': '{output_text}'}}")
    
    return tracer

tracer = finish_request(tracer)

## 7.9 完整流程图

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                              请求生命周期                                    │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  用户                API Server              Tokenizer            Scheduler  │
│   │                     │                      │                     │       │
│   │  POST /v1/chat      │                      │                     │       │
│   │ ─────────────────► │                      │                     │       │
│   │                     │   TokenizeMsg        │                     │       │
│   │                     │ ────────────────────►│                     │       │
│   │                     │                      │   UserMsg           │       │
│   │                     │                      │ ───────────────────►│       │
│   │                     │                      │                     │       │
│   │                     │                      │    ┌────────────────┤       │
│   │                     │                      │    │ 1. 查询 Cache  │       │
│   │                     │                      │    │ 2. 分配资源    │       │
│   │                     │                      │    │ 3. Prefill     │       │
│   │                     │                      │    │ 4. Decode loop │       │
│   │                     │                      │    └────────────────┤       │
│   │                     │                      │                     │       │
│   │                     │                      │   DetokenizeMsg     │       │
│   │                     │                      │◄───────────────────│       │
│   │                     │   StreamResponse     │                     │       │
│   │                     │◄────────────────────│                     │       │
│   │  SSE chunks         │                      │                     │       │
│   │◄───────────────────│                      │                     │       │
│   │                     │                      │                     │       │
└──────────────────────────────────────────────────────────────────────────────┘
```

## 7.10 Overlap Scheduling 详解

重叠调度可以隐藏 CPU 调度开销。

In [ ]:
def visualize_overlap_scheduling():
    """可视化 Overlap Scheduling"""
    print("=== 普通调度 (无重叠) ===")
    print("")
    print("时间 →")
    print("CPU:  [调度Batch1][等待  ][调度Batch2][等待  ][调度Batch3]")
    print("GPU:  [  等待   ][Batch1][  等待   ][Batch2][  等待   ]")
    print("")
    print("问题: CPU 调度和 GPU 计算串行执行")
    
    print("\n" + "="*50)
    print("\n=== Overlap Scheduling (重叠调度) ===")
    print("")
    print("时间 →")
    print("CPU:  [调度B1][调度B2+处理B1结果][调度B3+处理B2结果]...")
    print("GPU:  [      ][     Batch1     ][     Batch2     ]...")
    print("             ▲                 ▲")
    print("             └── CPU 和 GPU 同时工作 ──┘")
    print("")
    print("优势: CPU 调度与 GPU 计算并行")

visualize_overlap_scheduling()

In [ ]:
def overlap_loop_pseudocode():
    """Overlap Loop 伪代码"""
    print("""
def overlap_loop(last_data):
    # 1. 接收新消息 (非阻塞，如果有正在处理的数据)
    blocking = (last_data is None) and (没有待处理的请求)
    for msg in receive_msg(blocking=blocking):
        process_one_msg(msg)  # 添加到等待队列
    
    # 2. 调度下一个批次
    forward_input = schedule_next_batch()
    ongoing_data = None
    
    if forward_input:
        # 3. 在 Engine 的 stream 中执行 (GPU)
        with engine_stream:
            engine.stream.wait_stream(scheduler.stream)
            ongoing_data = (forward_input, engine.forward_batch(...))
    
    # 4. 处理上一个批次的结果 (CPU, 与 GPU 并行)
    process_last_data(last_data, ongoing_data)
    
    return ongoing_data
""")

overlap_loop_pseudocode()

## 7.11 小结

### 请求生命周期:

1. **API Server**: 接收 HTTP 请求，分配 UID
2. **Tokenizer**: 文本 → Token IDs
3. **Scheduler**: 查询缓存，分配资源，加入队列
4. **Prefill**: 处理输入，生成第一个 token
5. **Decode**: 逐步生成后续 tokens
6. **完成**: 释放资源，返回结果

### 关键优化:

- **Radix Cache**: 复用相同前缀的 KV Cache
- **CUDA Graph**: 加速 Decode 阶段
- **Overlap Scheduling**: 隐藏 CPU 开销
- **Continuous Batching**: 动态调整批次

---

**下一步**: [Module 8: 高级优化](./08_advanced_optimizations.ipynb) - 深入学习 CUDA Graph 和 Tensor Parallelism。